# 01 — Projections: column lineage 1:1

Простейший случай column lineage: каждая выходная колонка происходит из ровно одной входной без преобразований (transformation type = `IDENTITY`).

**Что увидим в Marquez:** у `default.stg_customers_basic` в column lineage tab появятся стрелки `raw_customers.id → stg_customers_basic.id`, `raw_customers.name → stg_customers_basic.name`, `raw_customers.country → stg_customers_basic.country`.

Prerequisite: прогнан `00_setup.ipynb`. **Restart Jupyter kernel** перед запуском.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('lineage_01_projections')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_01_projections', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:07:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:07:22 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:07:33 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:07:33 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:07:33 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.


app name: lineage_01_projections


In [3]:
df = spark.sql('''
    SELECT id, name, country
    FROM default.raw_customers
''')
df.write.mode('overwrite').saveAsTable('default.stg_customers_basic')
spark.table('default.stg_customers_basic').show(5, truncate=False)

26/05/27 13:08:22 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+---+------+-------+
|id |name  |country|
+---+------+-------+
|0  |user_0|RU     |
|1  |user_1|US     |
|2  |user_2|DE     |
|3  |user_3|FR     |
|4  |user_4|RU     |
+---+------+-------+
only showing top 5 rows



In [4]:
spark.stop()

## Что смотреть

1. Marquez UI → namespace `hadoop-cluster` → datasets → `default.stg_customers_basic` → tab **Lineage**
2. API: `curl 'http://localhost:5000/api/v1/namespaces/hadoop-cluster/datasets/default.stg_customers_basic'`
3. Должно быть 3 column-level ребра, все с transformation type `IDENTITY` (или эквивалентным).